# Threshold sweep training (GPC) — Colab
1. **Cell 1**: settings (form on the right) + upload the PML CSV  
2. **Cell 2**: functions  
3. **Cell 3**: internal CV + calibration + threshold sweep (PML only). Each fold is checkpointed (resume works only while the same Colab session is alive).  
4. **Cell 4**: final model + download the `.joblib` and a zip of the results

Choose the threshold criterion **before** running. Never choose it from UKB.

In [ ]:
#@title 1) Settings + upload the PML CSV
#@markdown **Training**
N_TRAIN_TOTAL = 1000  #@param {type:"integer"}
N_FOLDS = 5  #@param {type:"integer"}
CV_RESTARTS = 0  #@param {type:"integer"}
FINAL_RESTARTS = 1  #@param {type:"integer"}
#@markdown **Threshold criterion (decide NOW)**
CRITERION = "f1"  #@param ["f1", "mcc", "youden", "sens"]
TARGET_SENSITIVITY = 0.80  #@param {type:"number"}
THR_START = 0.02  #@param {type:"number"}
THR_STOP = 0.95  #@param {type:"number"}
THR_STEP = 0.01  #@param {type:"number"}

import os
from pathlib import Path

from google.colab import files

BASE = Path("/content/TdP")
BASE.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
print("Choose the PML CSV file to upload:")
up = files.upload()
DATA_NAME = list(up.keys())[0]

CONFIG = {
    "data_name": DATA_NAME,
    "label_column": "outcome",
    "positive_class": "at risk",
    "features": [
        "sum_of_purple_area", "sum_of_blue_area", "sum_of_green_area",
        "sum_of_lime_area", "sum_of_yellow_area", "sum_of_orange_area",
        "sum_of_dark_orange_area", "sum_of_red_area", "sum_of_dark_red_area",
        "First_derivative_1", "First_derivative_2", "First_derivative_3",
        "First_derivative_4", "First_derivative_5", "First_derivative_6",
        "First_derivative_7", "First_derivative_8",
        "Purple_Blue_Green", "Blue_Green_Lime", "Green_Lime_Yellow",
        "Lime_Yellow_Orange", "Yellow_Orange_DarkOrange",
        "Orange_DarkOrange_Red", "DarkOrange_Red_DarkRed",
        "inflectionPoint",
    ],
    "n_train_total": N_TRAIN_TOTAL,
    "n_folds": N_FOLDS,
    "cv_restarts": CV_RESTARTS,
    "final_restarts": FINAL_RESTARTS,
    "criterion": CRITERION,
    "target_sensitivity": TARGET_SENSITIVITY,
    "thr_start": THR_START, "thr_stop": THR_STOP, "thr_step": THR_STEP,
    "max_iter_predict": 300,
    "random_state": 42,
}
print("BASE:", BASE)
print({k: v for k, v in CONFIG.items() if k != "features"})


In [ ]:
#@title 2) Functions
import time, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import sklearn

warnings.filterwarnings("ignore", category=ConvergenceWarning)
_trapz = getattr(np, "trapezoid", None) or np.trapz
print("scikit-learn", sklearn.__version__)


def find_file(name):
    p = BASE / name
    if p.exists():
        return p
    hits = list(BASE.rglob(name))
    if not hits:
        raise FileNotFoundError(f"File {name} not found under {BASE}")
    return hits[0]


def read_smart(path, required):
    for header in [0, 1, 2, 3]:
        for sep in [";", ",", "\t"]:
            for enc in ["utf-8-sig", "latin-1"]:
                try:
                    probe = pd.read_csv(path, sep=sep, header=header, nrows=5, encoding=enc)
                except Exception:
                    continue
                cols = [str(c).strip() for c in probe.columns]
                if all(c in cols for c in required):
                    df = pd.read_csv(path, sep=sep, header=header, encoding=enc)
                    df.columns = [str(c).strip() for c in df.columns]
                    return df
    raise ValueError("Required columns not found in the file")


def pr_auc_dg(y_true, p):
    """PR-AUC (Davis & Goadrich interpolation)."""
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    P = int(y_true.sum())
    if P == 0:
        return np.nan
    order = np.argsort(-p, kind="mergesort")
    ys, ps = y_true[order], p[order]
    tps, fps = np.cumsum(ys), np.cumsum(1 - ys)
    idx = np.r_[np.where(np.diff(ps))[0], len(ys) - 1]
    tps = np.r_[0, tps[idx]].astype(float)
    fps = np.r_[0, fps[idx]].astype(float)
    area = 0.0
    for i in range(1, len(tps)):
        dTP = tps[i] - tps[i - 1]
        if dTP <= 0:
            continue
        skew = (fps[i] - fps[i - 1]) / dTP
        xs = np.arange(0, int(dTP) + 1, dtype=float)
        num = tps[i - 1] + xs
        den = num + fps[i - 1] + skew * xs
        prec = np.divide(num, den, out=np.ones_like(num), where=den > 0)
        if num[0] == 0 and len(prec) > 1:
            prec[0] = prec[1]
        area += _trapz(prec, num / P)
    return float(area)


def build_gpc(n_features, restarts, seed, max_iter):
    kernel = (ConstantKernel(1.0)
              * Matern(length_scale=np.ones(n_features), nu=2.5)
              + WhiteKernel(1e-3))
    return GaussianProcessClassifier(kernel=kernel, n_restarts_optimizer=restarts,
                                     max_iter_predict=max_iter, random_state=seed)


def enriched_sample(idx, y, n_total, rng):
    pos = idx[y[idx] == 1]
    neg = idx[y[idx] == 0]
    n_neg = min(len(neg), max(0, n_total - len(pos)))
    return np.r_[pos, rng.choice(neg, n_neg, replace=False)]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def fit_platt(p_raw, y):
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_raw).reshape(-1, 1), y)
    return float(lr.coef_[0][0]), float(lr.intercept_[0])


def apply_calibration(p_raw, cal):
    """Same formula as External_Cohort.py."""
    return sigmoid(cal["a"] * (logit(p_raw) - cal.get("prior_shift", 0.0)) + cal["b"])


def metrics_at(y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / den if den > 0 else 0.0
    return {"threshold": round(float(t), 4), "Sensitivity": sens, "Specificity": spec,
            "PPV": ppv, "NPV": npv, "F1": f1, "MCC": mcc,
            "Youden": sens + spec - 1, "Balanced_Acc": (sens + spec) / 2,
            "Flagged_%": 100 * pred.mean(),
            "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}


def sweep(y, p, cfg):
    grid = np.round(np.arange(cfg["thr_start"], cfg["thr_stop"] + 1e-9, cfg["thr_step"]), 4)
    return pd.DataFrame([metrics_at(y, p, t) for t in grid])


def pick_best(table, cfg):
    c = cfg["criterion"]
    if c == "sens":
        ok = table[table["Sensitivity"] >= cfg["target_sensitivity"]]
        if ok.empty:
            print(f"  No threshold reaches Sensitivity >= {cfg['target_sensitivity']}; "
                  f"using the lowest threshold.")
            return table.iloc[0]
        return ok.sort_values("threshold").iloc[-1]
    col = {"f1": "F1", "mcc": "MCC", "youden": "Youden"}[c]
    return table.loc[table[col].idxmax()]

print("Functions ready.")


In [ ]:
#@title 3) Data + internal CV (resumable) + calibration + threshold sweep
cfg = CONFIG
feats, label = cfg["features"], cfg["label_column"]
t_start = time.time()

path = find_file(cfg["data_name"])
print("Data:", path)
df = read_smart(path, feats + [label])
df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=feats + [label]).reset_index(drop=True)
X = df[feats].values.astype(float)
y = (df[label].astype(str).str.strip().str.lower()
     == cfg["positive_class"].lower()).astype(int).values
n_pos = int(y.sum())
print(f"PML: {len(y)} rows | at risk: {n_pos} | controls: {len(y) - n_pos}")

n_tot = cfg["n_train_total"]
if n_tot <= n_pos:
    n_tot = n_pos + 500
    print(f"n_train_total must be > {n_pos}; using {n_tot}")
    cfg["n_train_total"] = n_tot

tag = f"n{n_tot}_{cfg['criterion']}_k{cfg['n_folds']}"
out_model = BASE / f"saved_models/internal_model_{tag}.joblib"
out_dir = BASE / f"results/threshold_sweep_{tag}"
ckpt_dir = out_dir / "fold_checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
out_model.parent.mkdir(parents=True, exist_ok=True)
print(f"Output model : {out_model}\nOutput folder: {out_dir}\n")

skf = StratifiedKFold(n_splits=cfg["n_folds"], shuffle=True, random_state=cfg["random_state"])
p_oof = np.full(len(y), np.nan)
cv_prev = []
print("=" * 70); print(f"Internal validation ({cfg['n_folds']}-fold)"); print("=" * 70)
for fold, (tr, va) in enumerate(skf.split(X, y)):
    ck = ckpt_dir / f"fold{fold}.npz"
    if ck.exists():                                   # resume after a disconnect
        z = np.load(ck)
        p_oof[va] = z["p"]; cv_prev.append(float(z["prev"]))
        print(f"fold={fold} loaded from checkpoint "
              f"ROC={roc_auc_score(y[va], p_oof[va]):.4f}")
        continue
    rng = np.random.default_rng(cfg["random_state"] + fold)
    sub = enriched_sample(tr, y, n_tot, rng)
    scaler = StandardScaler().fit(X[tr])
    gpc = build_gpc(len(feats), cfg["cv_restarts"], cfg["random_state"], cfg["max_iter_predict"])
    t0 = time.time()
    gpc.fit(scaler.transform(X[sub]), y[sub])
    p_oof[va] = gpc.predict_proba(scaler.transform(X[va]))[:, 1]
    prev = float(y[sub].mean()); cv_prev.append(prev)
    np.savez(ck, p=p_oof[va], prev=prev)
    print(f"fold={fold} train={len(sub)} (at risk={y[sub].sum()}, {prev:.1%}) "
          f"ROC={roc_auc_score(y[va], p_oof[va]):.4f} PR={pr_auc_dg(y[va], p_oof[va]):.4f} "
          f"({(time.time() - t0) / 60:.1f} min)", flush=True)

# Platt calibration on OOF
a, b = fit_platt(p_oof, y)
p_cal = apply_calibration(p_oof, {"a": a, "b": b})
print("\n" + "=" * 70); print("OOF summary"); print("=" * 70)
print(f"ROC AUC     : {roc_auc_score(y, p_oof):.4f}")
print(f"PR-AUC (DG) : {pr_auc_dg(y, p_oof):.4f}  (baseline {y.mean():.4f})")
print(f"Platt a={a:.4f} b={b:.4f} | mean p_cal={p_cal.mean():.4f} | observed={y.mean():.4f}")
print(f"Brier (cal) : {brier_score_loss(y, p_cal):.4f}")
print(f"Calibrated OOF quantiles: 50%={np.quantile(p_cal, .5):.4f}  "
      f"90%={np.quantile(p_cal, .9):.4f}  99%={np.quantile(p_cal, .99):.4f}  max={p_cal.max():.4f}")

# threshold sweep (PML only)
table = sweep(y, p_cal, cfg)
best = pick_best(table, cfg)
thr = float(best["threshold"])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
show = ["threshold", "Sensitivity", "Specificity", "PPV", "NPV", "F1", "MCC",
        "Youden", "Flagged_%", "TP", "FP", "FN"]
print("\n" + "=" * 70)
print(f"Threshold sweep {cfg['thr_start']}-{cfg['thr_stop']} on calibrated OOF (PML)")
print("=" * 70)
print(table[show].round(4).to_string(index=False))
print(f"\n>>> Best threshold by '{cfg['criterion']}': {thr:.2f}")
print(best[show].round(4).to_string())
table.to_csv(out_dir / "threshold_sweep_PML_oof.csv", index=False)
df.assign(true_label=y, p_raw_oof=p_oof, p_cal_oof=p_cal).to_csv(
    out_dir / "oof_predictions.csv", index=False)
print("\nSaved sweep + OOF to", out_dir)


In [ ]:
#@title 4) Final model + save .joblib
print("=" * 70); print(f"Training final model (restarts={cfg['final_restarts']}) ..."); print("=" * 70)
rng = np.random.default_rng(cfg["random_state"] + 100)
sub = enriched_sample(np.arange(len(y)), y, n_tot, rng)
scaler = StandardScaler().fit(X)
gpc = build_gpc(len(feats), cfg["final_restarts"], cfg["random_state"], cfg["max_iter_predict"])
t0 = time.time()
gpc.fit(scaler.transform(X[sub]), y[sub])
print(f"Done in {(time.time() - t0) / 60:.1f} min | train={len(sub)} (at risk={y[sub].sum()})")
print("Kernel:", gpc.kernel_)

# prior shift: final model sees a higher at-risk share than the CV models
prev_cv, prev_final = float(np.mean(cv_prev)), float(y[sub].mean())
shift = float(logit(prev_final) - logit(prev_cv))
calibration = {"method": "platt", "a": a, "b": b, "prior_shift": shift,
               "prior_shift_note": f"logit({prev_final:.4f}) - logit({prev_cv:.4f})"}
print(f"prev_cv={prev_cv:.4f} | prev_final={prev_final:.4f} | prior_shift={shift:.4f}")

bundle = {
    "model": gpc, "scaler": scaler, "feature_columns": feats,
    "threshold": thr,                      # applied to the CALIBRATED probability
    "positive_class": cfg["positive_class"],
    "calibration": calibration,
    "training_rows": len(y),
    "train_prevalence": prev_final,
    "threshold_sweep_pml": table.to_dict(orient="list"),
    "sklearn_version": sklearn.__version__,
    "config": {**cfg, "train_samples": n_tot, "use_ard": True,
               "n_restarts_optimizer": cfg["final_restarts"],
               "sampling": "all_positives_plus_random_negatives", "threshold": thr},
    "internal_cv": {"roc_auc": roc_auc_score(y, p_oof), "pr_auc": pr_auc_dg(y, p_oof),
                    **{k: best[k] for k in show[1:]}},
}
joblib.dump(bundle, out_model)
print(f"\nSaved model : {out_model}")
print(f"Total time  : {(time.time() - t_start) / 60:.0f} min")

# download the model + results (Colab files are deleted when the session ends)
import shutil
zip_path = shutil.make_archive(str(BASE / f"results_{tag}"), "zip", root_dir=out_dir)
files.download(str(out_model))
files.download(zip_path)
